# PHASE 5 — KAGGLE BACKEND: PPO SURGICAL TRAJECTORY TRAINING
```
=========================================================
  Thesis: 3D Resection & Surgical Planning of Brain Tumors
  Module: Reinforcement Learning — Training Backend
  Hardware: Kaggle T4 / P100 GPU (16 GB VRAM)
  Output: ppo_best_model.zip  |  vecnorm.pkl  |  metadata.json
=========================================================
```
**Pipeline position:**
```
Phase 4 (.npz) ──▶  [THIS NOTEBOOK]  ──▶  ppo_best_model.zip
                         │                      │
                   DummyVecEnv PPO        Colab Frontend
                   500 k steps            (Phase 6 viz)
```
**Why DummyVecEnv, not SubprocVecEnv?**
Kaggle Jupyter kernels share a CUDA context with forked Python workers.  
`SubprocVecEnv(start_method='fork')` causes a deadlock when the child  
process tries to initialise CUDA after the parent has already locked  
the device. `DummyVecEnv` runs everything in a single process, no fork,  
no deadlock — and on a T4 GPU the throughput difference is negligible  
because the Gymnasium step itself is CPU-bound (NumPy only).

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 0 — INSTALL DEPENDENCIES                                   ║
# ║  Runtime: ~90 s (Kaggle caches wheels between sessions)          ║
# ╚═══════════════════════════════════════════════════════════════════╝

!pip install -q 'stable-baselines3[extra]>=2.3.0' 'gymnasium>=0.29.0'
!pip install -q scipy scikit-image nibabel

import os, json, time, glob, warnings
import numpy as np
import torch
from scipy.ndimage import distance_transform_edt, gaussian_filter
from skimage.transform import resize

import gymnasium as gym
from gymnasium import spaces
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecMonitor
from stable_baselines3.common.monitor   import Monitor
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.utils     import set_random_seed

warnings.filterwarnings('ignore')
set_random_seed(42)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = '/kaggle/working'

print(f'PyTorch  : {torch.__version__}')
print(f'SB3      : {sb3.__version__}')
print(f'Device   : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('[OK] Dependencies loaded.')

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — DATA LOADING + SELF-HEALING PREPROCESSING             ║
# ║                                                                   ║
# ║  Problem: Phase 3's SafetyMapNetwork may produce an              ║
# ║  uncalibrated sigmoid output where all values cluster in         ║
# ║  [0.6, 0.9] due to BCE loss with imbalanced labels.             ║
# ║  This creates a near-flat reward landscape for the RL agent.    ║
# ║                                                                   ║
# ║  Fix: Dynamic contrast stretching (Min-Max stretch to [0,1])    ║
# ║  followed by EDT recalculation at the new P < 0.25 threshold.   ║
# ╚═══════════════════════════════════════════════════════════════════╝

# ── EDIT: set to your Phase 4 output path ─────────────────────────
NPZ_SEARCH_PATHS = [
    '/kaggle/input/**/*continuous_safety*.npz',
    '/kaggle/input/**/*safety*.npz',
    '/kaggle/working/*safety*.npz',
]
# ──────────────────────────────────────────────────────────────────

MAX_VOL_DIM = 80   # Downsample to this if volume is larger (memory budget)


def generate_synthetic_context(shape=(80, 80, 80)) -> dict:
    """
    Synthetic fallback — creates a medically plausible environment:
    - Ellipsoidal tumor at centre
    - Two vascular danger cylinders nearby
    - Gaussian safety decay from tumor boundary
    - EDT distance buffer
    Used when no Phase 4 .npz is found (demo / unit-test mode).
    """
    print('[SYNTHETIC] Generating fallback surgical context...')
    D, H, W = shape
    ctr = np.array([D//2, H//2, W//2])
    z, y, x = np.mgrid[0:D, 0:H, 0:W]

    # Irregular ellipsoidal tumour
    rng = np.random.RandomState(42)
    dist = np.sqrt((z-ctr[0])**2/64 + (y-ctr[1])**2/81 + (x-ctr[2])**2/49)
    noise = rng.randn(*shape) * 0.15
    tumor = (dist + noise) < 1.0

    # Two vascular danger cylinders
    v1 = (abs(x - ctr[2] - 12) < 3) & (abs(y - ctr[1] - 8) < 3)
    v2 = (abs(x - ctr[2] + 10) < 2) & (abs(z - ctr[0] + 5) < 10)

    # Safety gradient: sigmoid decay + vessel knock-outs
    edt_bg = distance_transform_edt(~tumor)
    sg = 1.0 / (1.0 + np.exp(-(edt_bg - 6.0) / 3.0))
    sg[v1 | v2] = 0.0
    sg = gaussian_filter(sg.astype(np.float32), sigma=1.5)

    danger  = sg < 0.25
    edt_map = distance_transform_edt(~danger).astype(np.float32)
    entry   = np.array([ctr[0], ctr[1], W - 5])

    print(f'  Shape        : {shape}')
    print(f'  Tumour voxels: {tumor.sum()}')
    print(f'  Safety range : [{sg.min():.3f}, {sg.max():.3f}]')
    return dict(safety_gradient=sg, physical_distance=edt_map,
                tumor_mask=tumor, centroid=ctr, entry_point=entry)


def load_and_heal_context(search_paths: list, max_dim: int = 80) -> dict:
    """
    Loads Phase 4 .npz and applies self-healing preprocessing.

    Self-Healing Steps
    ------------------
    1. Min-Max contrast stretch: forces safety_gradient ∈ [0, 1].
       Handles uncalibrated sigmoid outputs from Phase 3 BCE loss.
    2. EDT recalculation: recomputes physical_distance using the
       corrected gradient (threshold P < 0.25 = severe danger).
    3. Volume downsampling: if any dim > max_dim, trilinearly
       resample to max_dim isotropic to fit RL within GPU memory.
    """
    # Auto-discover the .npz file
    npz_path = None
    for pattern in search_paths:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            npz_path = hits[0]
            break

    if npz_path is None:
        print('[WARN] No .npz found — using synthetic context.')
        return generate_synthetic_context()

    print(f'[LOAD] Found: {npz_path}')
    raw = np.load(npz_path, allow_pickle=True)

    sg  = raw['safety_gradient'].astype(np.float32)
    tm  = raw['tumor_mask'].astype(bool)
    ctr = raw['centroid'].astype(int)
    ep  = raw['entry_point'].astype(int)

    # ── Step 1: Min-Max contrast stretch ──────────────────────────
    sg_min, sg_max = sg.min(), sg.max()
    spread = sg_max - sg_min
    if spread < 0.5:   # Compressed gradient — heal it
        print(f'[HEAL] Safety gradient compressed [{sg_min:.3f}, {sg_max:.3f}]')
        print(f'[HEAL] Applying Min-Max contrast stretch → [0.0, 1.0]')
        sg = (sg - sg_min) / (spread + 1e-8)
    else:
        print(f'[OK]   Safety gradient range [{sg_min:.3f}, {sg_max:.3f}] — no stretch needed.')
    sg = np.clip(sg, 0.0, 1.0)

    # ── Step 2: Recalculate EDT from corrected gradient ───────────
    severe_danger = (sg < 0.25)
    if severe_danger.sum() == 0:
        print('[HEAL] No severe-danger voxels after stretch — using tumour boundary as EDT source.')
        severe_danger = tm   # Fallback: tumour IS the danger
    edt_map = distance_transform_edt(~severe_danger).astype(np.float32)
    print(f'[EDT]  Recalculated. Max distance: {edt_map.max():.1f} vox')

    # ── Step 3: Downsample if too large for RL ────────────────────
    orig_shape = sg.shape
    if max(orig_shape) > max_dim:
        tgt    = tuple(min(max_dim, s) for s in orig_shape)
        scale  = np.array(tgt, dtype=float) / np.array(orig_shape)
        print(f'[DOWN] {orig_shape} → {tgt}')
        sg      = resize(sg,              tgt, order=1, anti_aliasing=True).astype(np.float32)
        edt_map = resize(edt_map,         tgt, order=1, anti_aliasing=True).astype(np.float32)
        tm      = resize(tm.astype(float),tgt, order=0) > 0.5
        ctr     = np.clip((ctr * scale).astype(int), 0, np.array(tgt) - 1)
        ep      = np.clip((ep  * scale).astype(int), 0, np.array(tgt) - 1)

    print(f'[OK]   Context ready. Shape: {sg.shape} | Tumour voxels: {tm.sum()}')
    return dict(safety_gradient=sg, physical_distance=edt_map,
                tumor_mask=tm, centroid=ctr, entry_point=ep)


CTX = load_and_heal_context(NPZ_SEARCH_PATHS, MAX_VOL_DIM)

# Persist the healed context so the Colab frontend can use it
healed_path = os.path.join(OUTPUT_DIR, 'healed_surgical_context.npz')
np.savez_compressed(healed_path, **{k: v for k, v in CTX.items()})
print(f'[SAVE] Healed context → {healed_path}')

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — GYMNASIUM ENVIRONMENT                                  ║
# ║                                                                   ║
# ║  Kinematic Model: Spherical-coordinate rigid retractor tube.     ║
# ║                                                                   ║
# ║    Tip position T relative to burr-hole E:                       ║
# ║      T = E + R·[sin θ·cos φ,  sin θ·sin φ,  cos θ]             ║
# ║                                                                   ║
# ║    Action Δ = [Δθ, Δφ, ΔR] ∈ [-1,+1]³  (normalised)           ║
# ║    Scaled per step: Δθ_max=0.08 rad, Δφ_max=0.10 rad, ΔR=2.5mm║
# ║                                                                   ║
# ║  Observation (22-dim float32):                                   ║
# ║    [0:3]  normalised tip xyz                                     ║
# ║    [3:6]  normalised θ, φ, R                                     ║
# ║    [6]    safety gradient at tip                                  ║
# ║    [7]    EDT distance / R_max (clipped 0–1)                     ║
# ║    [8]    EOR progress [0–1]                                     ║
# ║    [9]    step fraction [0–1]                                    ║
# ║    [10:13] unit direction vector entry→tip                       ║
# ║    [13:16] previous action (angular velocity)                    ║
# ║    [16]   dist to centroid / R_max                               ║
# ║    [17:20] look-ahead probe safety (3 depths: 30/60/100% ahead)  ║
# ║    [20]   on-tumour binary flag                                   ║
# ║    [21]   padding zero                                           ║
# ╚═══════════════════════════════════════════════════════════════════╝

class ResectionEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self,
                 ctx          : dict,
                 max_steps    : int   = 400,
                 eor_target   : float = 0.85,
                 r_max        : float = 65.0,
                 dtheta_max   : float = 0.08,
                 dphi_max     : float = 0.10,
                 dr_max       : float = 2.5,
                 crit_safety  : float = 0.15,
                 w_eor        : float = 15.0,
                 w_risk       : float = 5.0,
                 w_smooth     : float = 0.5,
                 w_dist       : float = 0.3):

        super().__init__()
        self.sg        = ctx['safety_gradient'].astype(np.float32)
        self.edt       = ctx['physical_distance'].astype(np.float32)
        self.tumor     = ctx['tumor_mask'].astype(bool)
        self.centroid  = ctx['centroid'].astype(float)
        self.entry     = ctx['entry_point'].astype(float)
        self.shape     = np.array(self.sg.shape, dtype=float)

        self.max_steps   = max_steps
        self.eor_target  = eor_target
        self.r_max       = r_max
        self.dtheta_max  = dtheta_max
        self.dphi_max    = dphi_max
        self.dr_max      = dr_max
        self.crit_safety = crit_safety
        self.w_eor       = w_eor
        self.w_risk      = w_risk
        self.w_smooth    = w_smooth
        self.w_dist      = w_dist
        self.total_tumor = float(self.tumor.sum()) + 1e-8

        # Initialise spherical angles pointing toward centroid
        vec = self.centroid - self.entry
        d   = np.linalg.norm(vec) + 1e-8
        vn  = vec / d
        self._i_theta = float(np.arccos(np.clip(vn[2], -1, 1)))
        self._i_phi   = float(np.arctan2(vn[1], vn[0]) % (2 * np.pi))
        self._i_r     = float(d * 0.3)

        self.action_space      = spaces.Box(-1.0, 1.0, (3,), np.float32)
        self.observation_space = spaces.Box(-1.0, 2.0, (22,), np.float32)

        # Episode state (initialised in reset)
        self._theta = self._phi = self._r = 0.0
        self._tip   = np.zeros(3)
        self._step  = 0
        self._resected    = None
        self._prev_action = np.zeros(3)
        self._trajectory  = []

    # ── Private helpers ─────────────────────────────────────────────

    def _sph2cart(self, θ, φ, R) -> np.ndarray:
        return self.entry + R * np.array([
            np.sin(θ) * np.cos(φ),
            np.sin(θ) * np.sin(φ),
            np.cos(θ),
        ])

    def _sample(self, pos: np.ndarray, arr: np.ndarray, default=0.0) -> float:
        idx = np.clip(pos.astype(int), 0, np.array(arr.shape) - 1)
        try:
            return float(arr[idx[0], idx[1], idx[2]])
        except IndexError:
            return default

    def _build_obs(self) -> np.ndarray:
        tip_n   = self._tip / (self.shape + 1e-8)
        sph_n   = np.array([self._theta/(np.pi/2), self._phi/(2*np.pi), self._r/self.r_max])
        sg_tip  = self._sample(self._tip, self.sg, 0.5)
        edt_tip = self._sample(self._tip, self.edt, self.r_max)
        eor     = float(self._resected.sum()) / self.total_tumor
        step_n  = self._step / self.max_steps
        dv      = self._tip - self.entry
        dm      = np.linalg.norm(dv) + 1e-8
        dir_n   = dv / dm
        dist_c  = min(np.linalg.norm(self._tip - self.centroid) / (self.r_max + 1e-8), 1.0)
        probes  = [self._sample(self._tip + dir_n * 15*f, self.sg, 0.5) for f in [.3,.6,1.]]
        on_t    = float(self._sample(self._tip, self.tumor.astype(np.float32), 0.0))
        return np.array([
            *tip_n, *sph_n, sg_tip, min(edt_tip/self.r_max,1.), eor, step_n,
            *dir_n, *self._prev_action, dist_c, *probes, on_t, 0.0
        ], dtype=np.float32)

    # ── Gymnasium API ────────────────────────────────────────────────

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = np.random.default_rng(seed)
        self._theta = np.clip(self._i_theta + rng.uniform(-.08, .08), 0.01, np.pi/2-.01)
        self._phi   = (self._i_phi + rng.uniform(-.08, .08)) % (2*np.pi)
        self._r     = np.clip(self._i_r + rng.uniform(-2., 2.), 1., self.r_max)
        self._tip         = self._sph2cart(self._theta, self._phi, self._r)
        self._step        = 0
        self._resected    = np.zeros(self.tumor.shape, dtype=bool)
        self._prev_action = np.zeros(3)
        self._trajectory  = [self._tip.copy()]
        return self._build_obs(), {}

    def step(self, action: np.ndarray):
        action = np.clip(action, -1., 1.)
        dθ = action[0] * self.dtheta_max
        dφ = action[1] * self.dphi_max
        dR = action[2] * self.dr_max
        ang_acc = abs(action[0]-self._prev_action[0]) + abs(action[1]-self._prev_action[1])

        self._theta = np.clip(self._theta + dθ, 0.01, np.pi/2-.01)
        self._phi   = (self._phi + dφ) % (2*np.pi)
        self._r     = np.clip(self._r + dR, 1., self.r_max)
        new_tip     = self._sph2cart(self._theta, self._phi, self._r)

        sg_new  = self._sample(new_tip, self.sg, 0.5)
        edt_new = self._sample(new_tip, self.edt, 0.)

        # Sweep voxels along path segment (interpolated)
        prev_eor = float(self._resected.sum()) / self.total_tumor
        n_pts    = max(2, int(np.linalg.norm(new_tip - self._tip) * 2))
        for t in np.linspace(0, 1, n_pts):
            pt  = (self._tip + t*(new_tip - self._tip)).astype(int)
            pt  = np.clip(pt, 0, np.array(self.tumor.shape)-1)
            if self.tumor[pt[0], pt[1], pt[2]]:
                self._resected[pt[0], pt[1], pt[2]] = True
        curr_eor  = float(self._resected.sum()) / self.total_tumor
        delta_eor = curr_eor - prev_eor

        # Reward function
        r_eor    =  self.w_eor    * delta_eor
        r_risk   = -self.w_risk   * max(0., 1. - sg_new - 0.3)
        r_smooth = -self.w_smooth * ang_acc
        r_dist   = -self.w_dist   * max(0., np.linalg.norm(new_tip-self.centroid)/(self.r_max+1e-8) - 0.5)
        reward   = r_eor + r_risk + r_smooth + r_dist

        self._tip         = new_tip
        self._step       += 1
        self._prev_action = action.copy()
        self._trajectory.append(new_tip.copy())

        terminated = False
        truncated  = self._step >= self.max_steps
        info       = {'eor': curr_eor, 'safety': sg_new,
                      'edt_mm': edt_new, 'step': self._step}

        if curr_eor >= self.eor_target:
            reward += 100.0
            terminated = True
            info['outcome'] = 'EOR_TARGET_REACHED'
        elif sg_new < self.crit_safety and edt_new < 2.0:
            reward -= 50.0
            terminated = True
            info['outcome'] = 'CRITICAL_SAFETY_BREACH'
        else:
            info['outcome'] = 'RUNNING'

        return self._build_obs(), float(reward), terminated, truncated, info

    def get_trajectory(self) -> np.ndarray:
        return np.array(self._trajectory)

    def get_eor(self) -> float:
        return float(self._resected.sum()) / self.total_tumor


# ── Smoke test ───────────────────────────────────────────────────────
print('Smoke-testing environment...')
_e = ResectionEnv(CTX)
_obs, _ = _e.reset(seed=0)
assert _obs.shape == (22,), f'Bad obs shape: {_obs.shape}'
for _ in range(10):
    _obs, _r, _t, _tr, _i = _e.step(_e.action_space.sample())
print(f'  obs  shape : {_obs.shape}')
print(f'  reward     : {_r:.4f}')
print(f'  EOR after 10 steps: {_i["eor"]*100:.2f}%')
print('[OK] Environment verified.')
del _e

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — TRAINING CALLBACK                                      ║
# ╚═══════════════════════════════════════════════════════════════════╝

class SurgicalCallback(BaseCallback):
    """
    SB3 callback that:
    1. Logs EOR, min-safety, and outcome counts every `log_freq` steps.
    2. Saves the best model whenever mean EOR improves.
    3. Writes a running metrics JSON so you can tail it from a second
       Kaggle cell without interrupting training.
    """
    def __init__(self, save_path: str, log_freq: int = 10_000,
                 best_eor: float = 0.0, verbose: int = 1):
        super().__init__(verbose)
        self.save_path  = save_path
        self.log_freq   = log_freq
        self.best_eor   = best_eor
        self.eor_buf    = []
        self.safe_buf   = []
        self.outcomes   = {'EOR_TARGET_REACHED': 0,
                           'CRITICAL_SAFETY_BREACH': 0,
                           'RUNNING': 0}
        self.history    = []
        self.t0         = time.time()

    def _on_step(self) -> bool:
        for info in self.locals.get('infos', []):
            self.eor_buf.append(info.get('eor', 0.0))
            self.safe_buf.append(info.get('safety', 0.5))
            oc = info.get('outcome', 'RUNNING')
            self.outcomes[oc] = self.outcomes.get(oc, 0) + 1

        if self.num_timesteps % self.log_freq == 0 and self.eor_buf:
            n        = min(500, len(self.eor_buf))
            mean_eor = float(np.mean(self.eor_buf[-n:]))
            mean_sg  = float(np.mean(self.safe_buf[-n:]))
            elapsed  = time.time() - self.t0
            sps      = self.num_timesteps / (elapsed + 1e-8)
            eta      = (500_000 - self.num_timesteps) / (sps + 1e-8)

            row = {
                'step'   : self.num_timesteps,
                'eor_pct': round(mean_eor * 100, 2),
                'safety' : round(mean_sg, 4),
                'success': self.outcomes['EOR_TARGET_REACHED'],
                'breach' : self.outcomes['CRITICAL_SAFETY_BREACH'],
                'sps'    : round(sps, 1),
                'eta_min': round(eta / 60, 1),
            }
            self.history.append(row)

            print(f"  Step {self.num_timesteps:>7,} "
                  f"| EOR {mean_eor*100:.1f}% "
                  f"| Safety {mean_sg:.3f} "
                  f"| ✓ {self.outcomes['EOR_TARGET_REACHED']} "
                  f"| ✗ {self.outcomes['CRITICAL_SAFETY_BREACH']} "
                  f"| {sps:.0f} sps "
                  f"| ETA {eta/60:.0f} min")

            # Save best model if EOR improved
            if mean_eor > self.best_eor:
                self.best_eor = mean_eor
                self.model.save(os.path.join(self.save_path, 'ppo_best_model'))
                print(f'  ✓ NEW BEST saved — EOR {mean_eor*100:.1f}%')

            # Persist metrics JSON for monitoring
            with open(os.path.join(self.save_path, 'training_metrics.json'), 'w') as f:
                json.dump({'history': self.history, 'outcomes': self.outcomes}, f)

        return True


print('[OK] Callback defined.')

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — PPO TRAINING (DummyVecEnv, 500 k steps)               ║
# ║                                                                   ║
# ║  DummyVecEnv wraps N gym environments sequentially in one        ║
# ║  Python process. No forking → no CUDA deadlock.                  ║
# ║  N=4 gives ~4× throughput vs N=1 for short-episode envs.         ║
# ║  Runtime: ~20-35 min on T4, ~12-18 min on P100.                  ║
# ╚═══════════════════════════════════════════════════════════════════╝

TOTAL_STEPS = 500_000
N_ENVS      = 4        # All in one process — safe on Kaggle

# Build DummyVecEnv
def _make():
    env = ResectionEnv(CTX)
    return Monitor(env)

vec_env = DummyVecEnv([_make] * N_ENVS)
vec_env = VecNormalize(
    vec_env,
    norm_obs    = True,
    norm_reward = True,
    clip_obs    = 5.0,
    clip_reward = 10.0,
)
vec_env = VecMonitor(vec_env)

# PPO model
model = PPO(
    policy         = 'MlpPolicy',
    env            = vec_env,
    n_steps        = 1024,         # Rollout per env (total buffer = 4 × 1024)
    batch_size     = 256,
    n_epochs       = 10,
    gamma          = 0.99,
    gae_lambda     = 0.95,
    clip_range     = 0.2,
    ent_coef       = 0.01,         # Entropy bonus: encourages exploration
    vf_coef        = 0.5,
    max_grad_norm  = 0.5,
    learning_rate  = 3e-4,
    policy_kwargs  = dict(
        net_arch       = dict(pi=[256, 256, 128], vf=[256, 256, 128]),
        activation_fn  = torch.nn.Tanh,
    ),
    verbose        = 0,
    device         = DEVICE,
    seed           = 42,
)

n_params = sum(p.numel() for p in model.policy.parameters())
print(f'[PPO] Policy parameters : {n_params:,}')
print(f'[PPO] Total steps       : {TOTAL_STEPS:,}')
print(f'[PPO] Effective batch   : {N_ENVS * 1024:,}')
print(f'[PPO] Device            : {DEVICE}')

cb = SurgicalCallback(
    save_path = OUTPUT_DIR,
    log_freq  = 10_000,
)
ckpt_cb = CheckpointCallback(
    save_freq   = 100_000 // N_ENVS,
    save_path   = OUTPUT_DIR,
    name_prefix = 'ppo_ckpt',
)

print('\n' + '═'*60)
print('  TRAINING — PPO Surgical Resection Planner')
print('  Step       | EOR    | Safety | ✓ Success | ✗ Breach | ETA')
print('═'*60)

t_start = time.time()
model.learn(
    total_timesteps   = TOTAL_STEPS,
    callback          = [cb, ckpt_cb],
    progress_bar      = True,
    reset_num_timesteps = True,
)
t_total = time.time() - t_start

# ── Save final model + VecNormalize statistics ─────────────────────
final_model_path  = os.path.join(OUTPUT_DIR, 'ppo_best_model')
vecnorm_path      = os.path.join(OUTPUT_DIR, 'vecnorm.pkl')

model.save(final_model_path)
vec_env.save(vecnorm_path)

vec_env.close()

print(f'\n[DONE] Training complete in {t_total/60:.1f} min')
print(f'[SAVE] Model   → {final_model_path}.zip')
print(f'[SAVE] VecNorm → {vecnorm_path}')

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — EXPORT MANIFEST + METADATA                            ║
# ║                                                                   ║
# ║  Saves a metadata.json describing the training run so the        ║
# ║  Colab frontend can validate it loaded the right model.          ║
# ╚═══════════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt

# Load metrics
with open(os.path.join(OUTPUT_DIR, 'training_metrics.json')) as f:
    metrics = json.load(f)

history = metrics['history']

# ── Training curve ─────────────────────────────────────────────────
if history:
    steps   = [h['step']    for h in history]
    eors    = [h['eor_pct'] for h in history]
    safes   = [h['safety']  for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(steps, eors, color='#00ff88', lw=2)
    axes[0].axhline(85, color='red', ls='--', lw=1, label='85% target')
    axes[0].set_title('EOR (%) over Training Steps', fontsize=12)
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('EOR (%)')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(steps, safes, color='#44aaff', lw=2)
    axes[1].axhline(0.25, color='orange', ls='--', lw=1, label='Critical threshold')
    axes[1].set_title('Mean Safety Score over Training Steps', fontsize=12)
    axes[1].set_xlabel('Step'); axes[1].set_ylabel('Safety')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    curve_path = os.path.join(OUTPUT_DIR, 'training_curve.png')
    plt.savefig(curve_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'[SAVE] Training curve → {curve_path}')

# ── metadata.json ──────────────────────────────────────────────────
best_row  = max(history, key=lambda h: h['eor_pct']) if history else {}
meta = {
    'thesis'        : '3D Resection & Surgical Planning of Brain Tumors Using DL + RL',
    'phase'         : 5,
    'framework'     : f'stable-baselines3=={sb3.__version__}',
    'total_steps'   : TOTAL_STEPS,
    'n_envs'        : N_ENVS,
    'vec_env_type'  : 'DummyVecEnv',
    'best_eor_pct'  : best_row.get('eor_pct', 0.0),
    'best_safety'   : best_row.get('safety', 0.0),
    'training_min'  : round(t_total / 60, 1),
    'vol_shape'     : list(CTX['safety_gradient'].shape),
    'tumor_voxels'  : int(CTX['tumor_mask'].sum()),
    'outcomes'      : metrics['outcomes'],
    'files'         : {
        'model'    : 'ppo_best_model.zip',
        'vecnorm'  : 'vecnorm.pkl',
        'context'  : 'healed_surgical_context.npz',
        'metrics'  : 'training_metrics.json',
        'curve'    : 'training_curve.png',
    }
}
meta_path = os.path.join(OUTPUT_DIR, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('\n' + '═'*60)
print('  PHASE 5 KAGGLE BACKEND — OUTPUT MANIFEST')
print('═'*60)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        sz = os.path.getsize(fpath) / 1e6
        print(f'  {fname:<45} {sz:.2f} MB')
print('═'*60)
print()
print('  NEXT: Download these 4 files to Colab:')
print('    1. ppo_best_model.zip')
print('    2. vecnorm.pkl')
print('    3. healed_surgical_context.npz')
print('    4. metadata.json')
print()
print('  Then run: Phase6_Colab_Frontend.ipynb')